# 01-03 类型注解与 Pydantic

**为什么 Agent 开发必须掌握 Pydantic？**

- **LLM 结构化输出**: 让 LLM 输出 JSON，用 Pydantic 验证和解析
- **工具定义**: LangChain/LangGraph 用 Pydantic 定义工具的输入输出
- **数据验证**: Agent 的状态、记忆、配置都用 Pydantic 建模
- **FastAPI**: 如果你要部署 Agent 服务，Pydantic 是核心依赖

**本节目标**：
- 掌握 Python 类型注解
- 熟练使用 Pydantic v2 建模
- 实战：定义 LLM 结构化输出的 schema

---

## 1. Python 类型注解

In [ ]:
from __future__ import annotations
from typing import Optional, Union, Literal, Any

# 基础类型注解
def greet(name: str, age: int, active: bool = True) -> str:
    return f"你好 {name}，{age}岁，{'活跃' if active else '不活跃'}"

# 容器类型
def process_docs(
    docs: list[str],          # 字符串列表
    metadata: dict[str, Any], # 字典
    tags: set[str] | None,    # 可选 set（Python 3.10+ 语法）
) -> tuple[list[str], int]:
    return docs, len(docs)

# Literal 类型 —— 限制只能是特定值
def call_model(
    provider: Literal["openai", "anthropic", "dashscope"],
    model: str,
) -> str:
    return f"调用 {provider}/{model}"

print(greet("Wayne", 22))
print(call_model("openai", "gpt-4o-mini"))

## 2. Pydantic 基础

Pydantic 是 Python 最流行的数据验证库，LangChain、FastAPI 等都重度依赖它。

In [ ]:
from pydantic import BaseModel, Field, field_validator, model_validator
from datetime import datetime
from typing import Literal

# 基础 Pydantic 模型
class AdCreative(BaseModel):
    """广告素材模型"""
    title: str = Field(..., min_length=1, max_length=30, description="广告标题，不超过30字")
    body: str = Field(..., min_length=10, max_length=200, description="广告正文")
    call_to_action: str = Field(default="了解更多", description="行动召唤文案")
    ad_type: Literal["image", "video", "text"] = "image"
    score: float = Field(default=0.0, ge=0.0, le=10.0, description="质量分 0-10")
    
    @field_validator("title")
    @classmethod
    def title_no_special_chars(cls, v: str) -> str:
        forbidden = ['!', '！', '???']  # 广告法禁用词简化版
        for f in forbidden:
            if f in v:
                raise ValueError(f"标题不能包含 '{f}'")
        return v.strip()

# 正常创建
ad = AdCreative(
    title="B站游戏皮肤限时优惠",
    body="全新皮肤上线，限时折扣，快来体验沉浸式游戏体验",
    ad_type="video",
    score=8.5
)
print(ad)
print(f"JSON: {ad.model_dump_json(indent=2)}")

In [ ]:
# 验证错误示例
from pydantic import ValidationError

try:
    bad_ad = AdCreative(
        title="",  # 空标题
        body="短",  # 太短
        score=15.0  # 超出范围
    )
except ValidationError as e:
    print("验证错误:")
    for error in e.errors():
        print(f"  - {error['loc']}: {error['msg']}")

## 3. 核心实战：LLM 结构化输出

这是 Agent 开发中最常见的模式——让 LLM 返回结构化数据，而不是纯文本。

In [ ]:
import json
import sys
sys.path.insert(0, "..")
from utils.llm_client import call_llm

# 定义 Agent 分析广告的输出 Schema
class AdAnalysis(BaseModel):
    """广告质量分析结果"""
    score: float = Field(description="综合质量分 0-10")
    issues: list[str] = Field(default_factory=list, description="发现的问题")
    suggestions: list[str] = Field(default_factory=list, description="改进建议")
    approved: bool = Field(description="是否通过审核")
    rejection_reason: str | None = Field(default=None, description="如不通过，说明原因")

def analyze_ad_with_llm(ad_content: str) -> AdAnalysis:
    """用 LLM 分析广告内容，返回结构化结果"""
    schema = AdAnalysis.model_json_schema()
    
    prompt = f"""分析以下广告内容的质量，以 JSON 格式返回分析结果。

广告内容：
{ad_content}

请严格按照以下 JSON Schema 返回（不要有其他文字）：
{json.dumps(schema, ensure_ascii=False, indent=2)}
"""
    
    try:
        response = call_llm(prompt, provider="openai")
        # 解析 JSON 并用 Pydantic 验证
        data = json.loads(response.strip().strip("```json").strip("```"))
        return AdAnalysis(**data)
    except Exception as e:
        print(f"LLM 调用失败（可能未配置 key）: {e}")
        # 返回示例数据
        return AdAnalysis(
            score=7.5,
            issues=["标题偏长"],
            suggestions=["缩短标题至15字以内", "增加行动召唤词"],
            approved=True,
        )

# 测试
test_ad = "B站独家游戏皮肤，限时优惠50%！超多款式等你选，点击立即购买，数量有限，先到先得"
result = analyze_ad_with_llm(test_ad)
print(f"分析结果:")
print(f"  质量分: {result.score}")
print(f"  通过审核: {result.approved}")
print(f"  问题: {result.issues}")
print(f"  建议: {result.suggestions}")

## 4. Agent 状态建模

In [ ]:
from typing import Annotated
from pydantic import BaseModel, Field

# LangGraph 中 Agent 状态的典型写法
class AgentState(BaseModel):
    """广告素材生成 Agent 的状态"""
    # 输入
    product_info: str = Field(description="产品信息")
    target_audience: str = Field(default="年轻用户", description="目标受众")
    
    # 中间状态
    draft_creative: AdCreative | None = None
    analysis: AdAnalysis | None = None
    iteration_count: int = 0
    
    # 输出
    final_creative: AdCreative | None = None
    status: Literal["pending", "drafting", "reviewing", "approved", "rejected"] = "pending"
    
    @property
    def needs_revision(self) -> bool:
        return (
            self.analysis is not None 
            and not self.analysis.approved 
            and self.iteration_count < 3
        )

# 创建初始状态
state = AgentState(
    product_info="B站会员订阅服务，支持大会员和普通会员",
    target_audience="18-30岁动漫爱好者"
)
print(f"初始状态: {state.status}")
print(f"需要修改: {state.needs_revision}")
print(f"\n完整状态 JSON:\n{state.model_dump_json(indent=2)}")

## 5. `model_validate` 与 `model_dump`

In [ ]:
# 从 dict / JSON 字符串创建模型
data = {
    "title": "B站直播礼物活动",
    "body": "送礼物给主播，赢取限定周边，活动仅限本周",
    "ad_type": "video",
    "score": 9.0
}

# 从 dict 创建
ad1 = AdCreative.model_validate(data)
print(f"从 dict 创建: {ad1.title}")

# 从 JSON 字符串创建
json_str = '{"title": "限时折扣游戏装备", "body": "全服最低价，性价比超高的游戏装备等你来买"}'
ad2 = AdCreative.model_validate_json(json_str)
print(f"从 JSON 创建: {ad2.title}")

# 导出为 dict（exclude 某些字段）
export_data = ad1.model_dump(exclude={"score"})
print(f"导出（排除 score）: {export_data}")

## 总结

| Pydantic 功能 | Agent 开发场景 |
|---|---|
| `BaseModel` | 定义 Agent 状态、工具输入输出 |
| `Field(...)` | 约束 LLM 输出格式 |
| `field_validator` | 验证业务规则（广告法合规） |
| `model_json_schema()` | 生成 JSON Schema 给 LLM |
| `model_validate_json()` | 解析 LLM 的 JSON 输出 |
| `model_dump()` | 序列化状态到存储 |

**自检**: 能定义一个 `ToolCall` 模型（包含 tool_name, args, result, success 字段）并用 Pydantic 验证一个合法调用吗？

**Phase 1 完成！** 下一步：`02-database-sql/01_mysql_crud_joins.ipynb`